# Solutions: Unit Testing Exercises

This notebook provides step-by-step solutions for writing and running unit tests in your RAP pipeline using pytest. Each solution matches the corresponding exercise notebook and is designed for beginners.

## Exercise 1 Solution: Review and adapt an existing unit test

Open `tests/test_cleaning.py` and run the test using the following command in the terminal:
```cmd
pytest tests/test_cleaning.py
```
After the test passes, change the function `clean_health_data` in `src/python_rap_demo/cleaning.py ` to the following function, save the file and run the test again:

In [ ]:
def clean_health_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean health data by dropping rows with missing values in key columns.

    Args:
        df (pd.DataFrame): Raw health data.

    Returns:
        pd.DataFrame: Cleaned health data with no missing values in critical columns.
    """
    df = df.copy()

    # Drop rows with missing values in height_cm, weight_kg, or diagnosis columns
    df = df.dropna(subset=["height_cm", "weight_kg", "diagnosis"])

    # Fill missing smoker values with 'Yes'
    df["smoker"] = df["smoker"].fillna("Yes")

    # Ensure gender is uppercase
    df["gender"] = df["gender"].str.upper()

    return df

Notice how the check for the missing smoker value fails. The test checks the first column for a "smoker" value of "No", however the modified `clean_health_data` function fills missing smoker values with 'Yes', changing the smoker value in the first column to 'Yes', which causes the test to fail.

The test failing highlights the function to developers who can then check if the change was correct or not. If it was not, the developer can fix the error in the function. If it was, the unit test can be adapted to incorporate the change. In this case assume the change was correct. In order for the test to pass change the expected "smoker" value to "Yes" instead of "No".
The original assert statement looks like this:

In [ ]:
assert cleaned["smoker"].iloc[0] == "No"

The changed assert statement should look like this:

In [ ]:
assert cleaned["smoker"].iloc[0] == "Yes"

## Exercise 2 Solution: Write a simple unit test for a new function

Here are example unit tests for the `flag_missing` and `impute_by_group` functions in `src/python_rap_demo/cleaning.py`:

In [ ]:
# Walkthrough: Unit test for flag_missing
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))

from python_rap_demo.cleaning import flag_missing


def test_flag_missing():
    """
    Test flag_missing
    """
    df = pd.DataFrame({"height_cm": [170, None], "weight_kg": [70, None]})
    flagged = flag_missing(df, ["height_cm", "weight_kg"])
    # Check that the _imputed columns are correct
    assert flagged["height_cm_imputed"].tolist() == [False, True]
    print("height_cm_imputed test passed.")
    assert flagged["weight_kg_imputed"].tolist() == [False, True]
    print("weight_kg_imputed test passed.")


test_flag_missing()

In [ ]:
# Example: Unit test for impute_by_group

# Note: This test will not run unless impute_by_group has been entered into cleaning.py


import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))

from python_rap_demo.cleaning import impute_by_group


def test_impute_by_group():
    """
    Test impute_by_group.
    """
    df = pd.DataFrame({"height_cm": [170, None, 160], "gender": ["M", "F", "F"]})
    imputed = impute_by_group(df, "height_cm", "gender")
    # Check that missing value is imputed with group mean
    expected = [170, 160, 160]
    assert imputed.tolist() == expected
    print("impute_by_group test passed.")


test_impute_by_group()

## Exercise 3 Solution: Run your unit tests

Run the following command in your terminal:
```cmd
pytest tests
```
**Expected output:**
- All tests should pass. If a test fails, check the error message and fix your code or tests.

## Exercise 4 Solution: Stretch - Check test coverage

Run the following commands:
```cmd
pip install coverage
coverage run -m pytest tests
coverage report
```
**Expected output:**
- You will see a report showing the percentage of code covered by tests. Aim for high coverage, but focus on testing important logic.

## Exercise 5 Solution: Stretch - Try parameterisation in pytest

Here are examples using `@pytest.mark.parametrize` for `flag_missing` and `impute_by_group`. Parameterisation lets you run the same test with different inputs, making your tests more robust and easier to maintain.

In [ ]:
# Note: These tests will not run unless impute_by_group and flag_missing have been entered into cleaning.py

import os
import sys

import pandas as pd
import pytest

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))

from python_rap_demo.cleaning import flag_missing, impute_by_group

# Parameterised test for flag_missing


@pytest.mark.parametrize(
    "df,columns,expected_height_flags,expected_weight_flags",
    [
        # Test case 1: One missing value in each column
        (
            pd.DataFrame({"height_cm": [170, None], "weight_kg": [70, None]}),
            ["height_cm", "weight_kg"],
            [False, True],
            [False, True],
        ),
        # Test case 2: All missing in height, one missing in weight
        (
            pd.DataFrame({"height_cm": [None, None], "weight_kg": [None, 80]}),
            ["height_cm", "weight_kg"],
            [True, True],
            [True, False],
        ),
    ],
)
def test_flag_missing_param(df, columns, expected_height_flags, expected_weight_flags):
    """
    Test flag_missing with multiple input cases using parameterisation.
    """
    flagged = flag_missing(df, columns)
    # Check that the _imputed columns match expected flags
    assert flagged["height_cm_imputed"].tolist() == expected_height_flags
    assert flagged["weight_kg_imputed"].tolist() == expected_weight_flags


# Parameterised test for impute_by_group


@pytest.mark.parametrize(
    "df,col,group_col,expected",
    [
        # Test case 1: Impute missing height by gender group mean
        (
            pd.DataFrame({"height_cm": [170, None, 160], "gender": ["M", "F", "F"]}),
            "height_cm",
            "gender",
            [170, 160, 160],
        ),
        # Test case 2: All missing in one group, fallback to overall mean
        (
            pd.DataFrame({"height_cm": [None, None, 150], "gender": ["M", "M", "F"]}),
            "height_cm",
            "gender",
            [150, 150, 150],
        ),
    ],
)
def test_impute_by_group_param(df, col, group_col, expected):
    """
    Test impute_by_group with multiple input cases using parameterisation.
    """
    imputed = impute_by_group(df, col, group_col)
    # Check that imputed values match expected output
    assert imputed.tolist() == expected